# Session 9 Homework

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format, hour, round, to_date, when
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from pyspark.sql.functions import avg, count, stddev, sum
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window
from pathlib import Path
import shutil

### Start a Spark session

In [3]:
spark = (
    SparkSession.builder
    .appName("Session09Part03")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/17 18:26:05 WARN Utils: Your hostname, MARCOSSOTO, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/17 18:26:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 18:26:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/17 18:26:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Load the CSV using an explicit schema

In [4]:
schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("request_count", IntegerType(), True),
    StructField("error_count", IntegerType(), True),
    StructField("latency_ms", DoubleType(), True),
    StructField("bytes_in", DoubleType(), True),
    StructField("bytes_out", DoubleType(), True),
])

events_df = spark.read.option("header", True).schema(schema).csv("../datasets/service_events.csv")

### Print the schema, row count, and column names

In [5]:
events_df.printSchema()
events_df.show(5, truncate=False)

print("Rows:", events_df.count())
print("Columns:", events_df.columns)

root
 |-- event_id: integer (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- request_count: integer (nullable = true)
 |-- error_count: integer (nullable = true)
 |-- latency_ms: double (nullable = true)
 |-- bytes_in: double (nullable = true)
 |-- bytes_out: double (nullable = true)



+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+
|event_id|service        |region  |event_time         |request_count|error_count|latency_ms|bytes_in|bytes_out|
+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+
|1       |auth           |eu-west |2026-05-04 00:00:00|1240         |8          |82.4      |1.45E7  |3.82E7   |
|2       |payments       |eu-west |2026-05-04 00:00:00|860          |21         |146.2     |1.92E7  |4.41E7   |
|3       |search         |us-east |2026-05-04 01:00:00|2110         |13         |94.8      |2.87E7  |7.63E7   |
|4       |recommendations|us-east |2026-05-04 01:00:00|1680         |18         |132.5     |3.48E7  |9.11E7   |
|5       |auth           |ap-south|2026-05-04 02:00:00|980          |4          |78.1      |1.16E7  |3.01E7   |
+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---

### Add:
error_rate,
total_bytes,
traffic_mb,
latency_band,
event_date,
event_hour,
day_of_week

In [6]:
events_enriched_df = (
    events_df
    .withColumn("error_rate", when(col("request_count") > 0, col("error_count") / col("request_count")).otherwise(0))
    .withColumn("total_bytes", col("bytes_in") + col("bytes_out"))
    .withColumn("traffic_mb", col("total_bytes") / 1048576)
    .withColumn(
        "latency_band",
        when(col("latency_ms") < 100, "fast")
        .when(col("latency_ms") < 160, "normal")
        .otherwise("slow"),
    )
    .withColumn("event_date", to_date(col("event_time")))
    .withColumn("event_hour", hour(col("event_time")))
    .withColumn("day_of_week", date_format(col("event_time"), "E"))
)

In [7]:
events_enriched_df.show()

+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+--------------------+-----------+------------------+------------+----------+----------+-----------+
|event_id|        service|  region|         event_time|request_count|error_count|latency_ms|bytes_in|bytes_out|          error_rate|total_bytes|        traffic_mb|latency_band|event_date|event_hour|day_of_week|
+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+--------------------+-----------+------------------+------------+----------+----------+-----------+
|       1|           auth| eu-west|2026-05-04 00:00:00|         1240|          8|      82.4|  1.45E7|   3.82E7|0.006451612903225...|     5.27E7|50.258636474609375|        fast|2026-05-04|         0|        Mon|
|       2|       payments| eu-west|2026-05-04 00:00:00|          860|         21|     146.2|  1.92E7|   4.41E7| 0.02441860465116279|     6.33E7|60.367584228

### Register a temporary view named 'service_events_enriched'

In [8]:
events_enriched_df.createOrReplaceTempView("service_events_enriched")

### Write at least six Spark SQL queries

In [9]:
spark.sql("""
    SELECT
        service,
        ROUND(AVG(error_rate), 4) AS average_error_rate,
        ROUND(AVG(latency_ms), 2) AS average_latency_ms,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY service
    ORDER BY average_error_rate DESC
""").show()

+---------------+------------------+------------------+----------------+
|        service|average_error_rate|average_latency_ms|total_traffic_mb|
+---------------+------------------+------------------+----------------+
|       payments|            0.0243|            174.63|          407.89|
|recommendations|            0.0119|            140.75|          723.65|
|         search|            0.0057|             94.02|          614.26|
|           auth|            0.0051|             79.63|          305.56|
+---------------+------------------+------------------+----------------+



In [10]:
spark.sql("""
    SELECT
        event_hour,
        ROUND(AVG(error_rate), 4) AS average_error_rate,
        ROUND(AVG(latency_ms), 2) AS average_latency_ms,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY event_hour
    ORDER BY event_hour
""").show()

+----------+------------------+------------------+----------------+
|event_hour|average_error_rate|average_latency_ms|total_traffic_mb|
+----------+------------------+------------------+----------------+
|         0|            0.0143|            115.08|          226.69|
|         1|            0.0085|             115.1|          447.94|
|         2|            0.0151|            127.38|          200.94|
|         3|            0.0102|            130.38|          484.18|
|         9|            0.0148|            138.95|          285.82|
|        10|            0.0078|            106.68|          405.79|
+----------+------------------+------------------+----------------+



In [11]:
spark.sql("""
    SELECT
        region,
        ROUND(AVG(error_rate), 4) AS average_error_rate,
        ROUND(AVG(latency_ms), 2) AS average_latency_ms,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY region
    ORDER BY total_traffic_mb DESC
""").show()

+--------+------------------+------------------+----------------+
|  region|average_error_rate|average_latency_ms|total_traffic_mb|
+--------+------------------+------------------+----------------+
| us-east|            0.0117|            127.03|          733.76|
| eu-west|            0.0122|            122.73|          710.87|
|ap-south|            0.0114|            117.03|          606.73|
+--------+------------------+------------------+----------------+



In [12]:
spark.sql("""
    SELECT
        region,
        latency_band,
        ROUND(AVG(error_rate), 4) AS average_error_rate,
        ROUND(AVG(latency_ms), 2) AS average_latency_ms,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY region, latency_band
    ORDER BY latency_band, total_traffic_mb DESC
""").show()

+--------+------------+------------------+------------------+----------------+
|  region|latency_band|average_error_rate|average_latency_ms|total_traffic_mb|
+--------+------------+------------------+------------------+----------------+
| us-east|        fast|            0.0054|             86.88|          325.01|
|ap-south|        fast|            0.0049|             82.73|          268.27|
| eu-west|        fast|            0.0059|             87.27|          216.77|
| eu-west|      normal|            0.0163|            138.83|          362.49|
| us-east|      normal|            0.0112|            136.75|          244.33|
|ap-south|      normal|            0.0103|             125.2|          218.96|
| us-east|        slow|            0.0247|             197.6|          164.41|
| eu-west|        slow|            0.0148|             164.7|          131.61|
|ap-south|        slow|            0.0256|            177.45|           119.5|
+--------+------------+------------------+----------

In [13]:
spark.sql("""
    SELECT
        event_hour,
        latency_band,
        ROUND(AVG(error_rate), 4) AS average_error_rate,
        ROUND(AVG(latency_ms), 2) AS average_latency_ms,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY event_hour, latency_band
    ORDER BY latency_band, total_traffic_mb DESC
""").show()

+----------+------------+------------------+------------------+----------------+
|event_hour|latency_band|average_error_rate|average_latency_ms|total_traffic_mb|
+----------+------------+------------------+------------------+----------------+
|         1|        fast|            0.0058|             93.45|          203.61|
|        10|        fast|            0.0052|             88.15|          186.82|
|         9|        fast|            0.0049|              80.3|           121.4|
|         3|        fast|            0.0058|              99.2|          114.06|
|         0|        fast|            0.0059|              81.3|          102.71|
|         2|        fast|            0.0045|              77.3|           81.44|
|         1|      normal|            0.0112|            136.75|          244.33|
|         3|      normal|            0.0101|             128.8|          238.51|
|        10|      normal|            0.0103|             125.2|          218.96|
|         0|      normal|   

In [14]:
spark.sql("""
    SELECT
        service,
        latency_band,
        ROUND(AVG(error_rate), 4) AS average_error_rate,
        ROUND(AVG(latency_ms), 2) AS average_latency_ms,
        ROUND(SUM(traffic_mb), 2) AS total_traffic_mb
    FROM service_events_enriched
    GROUP BY service, latency_band
    ORDER BY latency_band, total_traffic_mb DESC
""").show()

+---------------+------------+------------------+------------------+----------------+
|        service|latency_band|average_error_rate|average_latency_ms|total_traffic_mb|
+---------------+------------+------------------+------------------+----------------+
|         search|        fast|            0.0056|             92.48|          504.49|
|           auth|        fast|            0.0051|             79.63|          305.56|
|recommendations|      normal|            0.0113|            135.96|          592.04|
|       payments|      normal|            0.0226|            148.85|          123.98|
|         search|      normal|            0.0065|             101.7|          109.77|
|       payments|        slow|            0.0252|            187.53|          283.91|
|recommendations|        slow|            0.0148|             164.7|          131.61|
+---------------+------------+------------------+------------------+----------------+



### Grouped summaries

In [15]:
service_summary_df = events_enriched_df.groupBy("day_of_week").agg(
    count("*").alias("total_records"),
    sum("request_count").alias("total_requests"),
    sum("error_count").alias("total_errors"),
    avg("error_rate").alias("average_error_rate"),
    avg("latency_ms").alias("average_latency_ms"),
    stddev("latency_ms").alias("latency_stddev"),
    sum("traffic_mb").alias("total_traffic_mb"),
)

service_summary_df.show()

+-----------+-------------+--------------+------------+--------------------+------------------+-----------------+------------------+
|day_of_week|total_records|total_requests|total_errors|  average_error_rate|average_latency_ms|   latency_stddev|  total_traffic_mb|
+-----------+-------------+--------------+------------+--------------------+------------------+-----------------+------------------+
|        Mon|           12|         17690|         179|0.011582802347575659|120.38333333333334|39.10695900251313|1007.3661804199219|
|        Tue|           12|         18440|         190|0.011955313352800827|124.13333333333333|44.25297798573629|1043.9872741699219|
+-----------+-------------+--------------+------------+--------------------+------------------+-----------------+------------------+



In [16]:
service_summary_df = events_enriched_df.groupBy("region").agg(
    count("*").alias("total_records"),
    sum("request_count").alias("total_requests"),
    sum("error_count").alias("total_errors"),
    avg("error_rate").alias("average_error_rate"),
    avg("latency_ms").alias("average_latency_ms"),
    stddev("latency_ms").alias("latency_stddev"),
    sum("traffic_mb").alias("total_traffic_mb"),
)

service_summary_df.show()

+--------+-------------+--------------+------------+--------------------+------------------+------------------+-----------------+
|  region|total_records|total_requests|total_errors|  average_error_rate|average_latency_ms|    latency_stddev| total_traffic_mb|
+--------+-------------+--------------+------------+--------------------+------------------+------------------+-----------------+
| eu-west|            8|         12630|         135| 0.01222987802843804|122.72500000000002| 35.19641824309481|710.8688354492188|
| us-east|            8|         13040|         135|0.011666829002573313|           127.025|49.296558848318355|733.7570190429688|
|ap-south|            8|         10460|          99|0.011410466519553372|           117.025| 42.02664970842056|606.7276000976562|
+--------+-------------+--------------+------------+--------------------+------------------+------------------+-----------------+



In [17]:
service_summary_df = events_enriched_df.groupBy("service").agg(
    count("*").alias("total_records"),
    sum("request_count").alias("total_requests"),
    sum("error_count").alias("total_errors"),
    avg("error_rate").alias("average_error_rate"),
    avg("latency_ms").alias("average_latency_ms"),
    stddev("latency_ms").alias("latency_stddev"),
    sum("traffic_mb").alias("total_traffic_mb"),
)

service_summary_df.show()

+---------------+-------------+--------------+------------+--------------------+------------------+------------------+-----------------+
|        service|total_records|total_requests|total_errors|  average_error_rate|average_latency_ms|    latency_stddev| total_traffic_mb|
+---------------+-------------+--------------+------------+--------------------+------------------+------------------+-----------------+
|           auth|            6|          7580|          39|0.005108422813243081| 79.63333333333334| 4.147609753420242|305.5572509765625|
|recommendations|            6|          9880|         119|0.011924216567787992|140.75000000000003| 16.61791202287459|723.6480712890625|
|       payments|            6|          5590|         136| 0.02432688314389142|174.63333333333333|22.910841683942277|407.8865051269531|
|         search|            6|         13080|          75|0.005716708875830474| 94.01666666666667| 5.659475829674217|614.2616271972656|
+---------------+-------------+----------

### Sorted rankings

In [18]:
traffic_window = Window.orderBy(col("total_traffic_mb").desc())
variability_window = Window.orderBy(col("latency_stddev").desc())
reliability_window = Window.orderBy(col("average_error_rate").asc())

ranked_summary_df = (
    service_summary_df
    .withColumn("traffic_rank", dense_rank().over(traffic_window))
    .withColumn("latency_variability_rank", dense_rank().over(variability_window))
    .withColumn("reliability_rank", dense_rank().over(reliability_window))
)

ranked_summary_df.orderBy("traffic_rank").show()

+---------------+-------------+--------------+------------+--------------------+------------------+------------------+-----------------+------------+------------------------+----------------+
|        service|total_records|total_requests|total_errors|  average_error_rate|average_latency_ms|    latency_stddev| total_traffic_mb|traffic_rank|latency_variability_rank|reliability_rank|
+---------------+-------------+--------------+------------+--------------------+------------------+------------------+-----------------+------------+------------------------+----------------+
|recommendations|            6|          9880|         119|0.011924216567787992|140.75000000000003| 16.61791202287459|723.6480712890625|           1|                       2|               3|
|         search|            6|         13080|          75|0.005716708875830474| 94.01666666666667| 5.659475829674217|614.2616271972656|           2|                       3|               2|
|       payments|            6|         

In [19]:
hourly_requests_df = events_enriched_df.groupBy("service", "event_hour").agg(
    sum("request_count").alias("hourly_requests")
)

hour_window = Window.partitionBy("service").orderBy(col("hourly_requests").desc())

busiest_hour_df = (
    hourly_requests_df
    .withColumn("hour_rank", dense_rank().over(hour_window))
    .filter(col("hour_rank") == 1)
    .select("service", col("event_hour").alias("busiest_hour"))
)

busiest_hour_df.show()

+---------------+------------+
|        service|busiest_hour|
+---------------+------------+
|           auth|           9|
|       payments|           9|
|recommendations|           3|
|         search|           3|
+---------------+------------+



### Final one-row-per-service summary with rank columns

In [20]:
final_summary_df = (
    ranked_summary_df
    .join(busiest_hour_df, on="service", how="left")
    .select(
        "service",
        "total_records",
        "total_requests",
        "total_errors",
        round("average_error_rate", 4).alias("average_error_rate"),
        round("average_latency_ms", 2).alias("average_latency_ms"),
        round("latency_stddev", 2).alias("latency_stddev"),
        round("total_traffic_mb", 2).alias("total_traffic_mb"),
        "busiest_hour",
        "traffic_rank",
        "latency_variability_rank",
        "reliability_rank",
    )
    .orderBy("traffic_rank")
)

final_summary_df.show(truncate=False)

+---------------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
|service        |total_records|total_requests|total_errors|average_error_rate|average_latency_ms|latency_stddev|total_traffic_mb|busiest_hour|traffic_rank|latency_variability_rank|reliability_rank|
+---------------+-------------+--------------+------------+------------------+------------------+--------------+----------------+------------+------------+------------------------+----------------+
|recommendations|6            |9880          |119         |0.0119            |140.75            |16.62         |723.65          |3           |1           |2                       |3               |
|search         |6            |13080         |75          |0.0057            |94.02             |5.66          |614.26          |3           |2           |3                       |2               |
|payments 

### Saving the final result

In [21]:
output_folder = Path("results/service_summary_spark")
single_csv = Path("results/service_summary.csv")

part_file = next(output_folder.glob("part-*.csv"))
shutil.copy(part_file, single_csv)

print(f"Saved {single_csv}")

Saved results/service_summary.csv


### Stop Spark at the end

In [22]:
spark.stop()